# Ablation Report Walkthrough

Phase 6 Increment 3 notebook for ablation package and manifest inspection.

Workflow:
1. Optionally regenerate ablation package artifacts.
2. Load latest template, markdown summary, and manifest.
3. Inspect publication tables and reproducibility metadata.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys

def resolve_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not resolve repository root")

ROOT = resolve_repo_root(Path.cwd())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

ROOT


In [ ]:
RUN_GENERATORS = False

if RUN_GENERATORS:
    env = dict(os.environ)
    env["PYTHONPATH"] = "src"
    command = [
        sys.executable,
        "scripts/run_ablation_report.py",
        "--config",
        "configs/experiments/ablation_report_baseline.json",
        "--containerized",
    ]
    print("RUN", " ".join(command))
    subprocess.run(command, cwd=ROOT, env=env, check=True)
else:
    print("RUN_GENERATORS is False; using existing ablation artifacts")


In [ ]:
ablation_root = ROOT / "artifacts" / "reports" / "ablation"
template_candidates = sorted(ablation_root.glob("ablation_report_template_*.json"), key=lambda path: path.stat().st_mtime)
manifest_candidates = sorted(ablation_root.glob("research_artifact_manifest_*.json"), key=lambda path: path.stat().st_mtime)
if not template_candidates or not manifest_candidates:
    raise FileNotFoundError("No ablation package artifacts found. Run scripts/run_ablation_report.py")

template_path = template_candidates[-1]
manifest_path = manifest_candidates[-1]

template = json.loads(template_path.read_text(encoding="utf-8"))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

template_path, manifest_path


In [ ]:
metadata = template["template_metadata"]
table_titles = [table["title"] for table in template["publication_tables"]]

overview = {
    "package_id": metadata["package_id"],
    "scenario_id": metadata["scenario_id"],
    "seed_count": metadata["seed_count"],
    "horizon": metadata["horizon"],
    "table_count": metadata["table_count"],
    "containerized": metadata["containerized"],
    "table_titles": table_titles,
}
overview


In [ ]:
primary_table = template["publication_tables"][0]
primary_headers = primary_table["columns"]
primary_rows = primary_table["rows"][:5]

print(" | ".join(primary_headers))
for row in primary_rows:
    print(" | ".join(str(item) for item in row))

manifest["manifest_metadata"]
